# 01 · Bronze ingestion
Raw CSV extracts -> `bronze.<entity>` Delta tables. Source values are kept as strings, with
`ingestion_timestamp`, `source_file` and `batch_id` added. Malformed rows go to
`quarantine.bronze_rejects`. Re-running a batch replaces that batch only (idempotent).

In [ ]:
%pip install -q pyyaml python-dateutil

In [ ]:
# Setup: make the repo's `src` package importable and select the Databricks environment.
# The notebook runs from <repo>/notebooks inside a Databricks Git folder.
import os, sys
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ["LAKEHOUSE_ENV"] = "databricks"

dbutils.widgets.text("catalog", "workspace")          # Free Edition default; production: real_estate
catalog = dbutils.widgets.get("catalog")
overrides = {"environments": {"databricks": {"catalog": catalog,
                                             "raw_volume_root": f"/Volumes/{catalog}/raw/landing"}}}

from src.pipeline import build_lakehouse
lh, cfg = build_lakehouse(env="databricks", overrides=overrides)
print(f"catalog={cfg.catalog}  as_of={cfg.reporting_as_of}  batches={list(cfg.batches)}")

## Create schemas and stage the raw files in a Unity Catalog Volume
Serverless compute reads files from Volumes (not DBFS). We copy the repo's `data/raw` folder into
`/Volumes/<catalog>/raw/landing`.

In [ ]:
import shutil
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.raw.landing")
lh.setup()   # bronze, silver, gold, quarantine, ops schemas
shutil.copytree(os.path.join(repo_root, "data", "raw"), cfg.raw_volume_root, dirs_exist_ok=True)
display(dbutils.fs.ls(cfg.raw_volume_root))

## Ingest batch_001 (initial load)

In [ ]:
from src.pipeline import run_bronze
results = run_bronze(lh, cfg, "batch_001")
display(spark.createDataFrame([{k: str(v) for k, v in r.items()} for r in results]))

In [ ]:
display(spark.table(f"{catalog}.quarantine.bronze_rejects"))

In [ ]:
display(spark.table(f"{catalog}.bronze.payments").limit(10))